# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, browse, and analyze the FAIR² mass spectrometry dataset using the `mlcroissant` library and the Croissant metadata schema.

### Dataset Source
The dataset is published in FAIR² and described via a Croissant schema accessible at the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
We'll load the Croissant metadata and access the dataset records with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print('Dataset loaded:')
print('Title:', metadata.name)
print('Version:', metadata.version)
print('Identifier:', metadata.identifier)
print('Published:', metadata.datePublished)
print('\nDescription:')
print(metadata.description)

## 2. Data Overview
We'll review available record sets, their IDs (`@id`), and enumerate available fields/columns and their `@id`s.

In [ ]:
# List all record sets and their field details
print('Available Record Sets:')
for rs in metadata.recordSets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    - Field @id: {field['@id']}   name: {field.get('name', 'N/A')}  type: {field.get('dataType', 'N/A')}")
    print()

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. All access is by `@id` as per Croissant and FAIR² best practices.

In [ ]:
# Collect all record set @ids
record_sets = [rs['@id'] for rs in metadata.recordSets]

# Preview the first record set and its field IDs
print('Record set IDs:', record_sets)
if record_sets:
    first_record_set = record_sets[0]
    print(f"\nColumns for first record set ({first_record_set}):")
    fields = None
    for rs in metadata.recordSets:
        if rs['@id'] == first_record_set:
            fields = rs.get('fields', [])
            break
    col_ids = [f['@id'] for f in fields] if fields else []
    print(col_ids)

    # Load records for first record set
    df = pd.DataFrame(list(dataset.records(record_set=first_record_set)))
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Choose a numeric field by its `@id` for filtering, normalizing and grouping. All entities referenced by their `@id`.

In [ ]:
# Replace with actual @ids based on the data overview above.
record_set_id = record_sets[0]
# Example numeric id: select '@id' of e.g. 'cr:peptideCount' if it exists
example_numeric_field = None
example_group_field = None
for rs in metadata.recordSets:
    if rs['@id'] == record_set_id:
        for f in rs.get('fields', []):
            if f.get('dataType', '').lower() in ('integer', 'float', 'number'):
                example_numeric_field = f['@id']
            # Try to pick a grouping field, e.g. accession or description
            if 'accession' in f.get('name', '').lower():
                example_group_field = f['@id']
        break

# Fallbacks if we didn't find something
if not example_numeric_field:
    example_numeric_field = df.select_dtypes(include=['number']).columns[0]
if not example_group_field:
    example_group_field = df.columns[0]

print("Using numeric field @id:", example_numeric_field)
print("Using group field @id:", example_group_field)

# Filter: take values above the median (safer than a fixed threshold)
if example_numeric_field in df.columns:
    threshold = df[example_numeric_field].median()
    filtered_df = df[df[example_numeric_field] > threshold].copy()
    print(f"Filtered records with {example_numeric_field} > median value ({threshold}):")
    print(filtered_df.head())

    # Z-score normalization
    filtered_df[f"{example_numeric_field}_normalized"] = (
        (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) /
        filtered_df[example_numeric_field].std()
    )
    print(f"\nNormalized {example_numeric_field} for filtered records:")
    print(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

    # Group by a group field (if categorical)
    if example_group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(example_group_field)[example_numeric_field].mean().reset_index()
        print(f"\nGrouped mean {example_numeric_field} by {example_group_field}:")
        print(grouped_df.head())

## 5. Visualization
Let's visualize distributions of the selected numeric field and its normalized values (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_numeric_field in df.columns:
    plt.figure(figsize=(10,4))
    sns.histplot(df[example_numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {example_numeric_field}')
    plt.xlabel(example_numeric_field)
    plt.show()

    if f"{example_numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{example_numeric_field}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f'Normalized Distribution of {example_numeric_field}')
        plt.xlabel(f"{example_numeric_field}_normalized")
        plt.show()

## 6. Conclusion
Using the Croissant schema and the `mlcroissant` library, we've loaded the FAIR² mass spectrometry dataset, explored record sets and fields by their `@id`, performed basic EDA and visualizations. The use of `@id` fields ensures transparent, reproducible references that work with FAIR data best-practices.

Further work may include more advanced normalization, protein-specific grouping, or cross-referencing with external databases using Croissant metadata.